# Find putative CPDs

## Rerun clean up commands

Since the ClinVar data was modified we begin by rerun the commands in the [clean_up.ipynb](clean_up.ipynb) notebook.

### Extract transcript IDs from variants

Pull the transcript record IDs from the ClinVar records of each gene.

**Output:** `vartranscripts` dictionary with genes as keys and lists of transcripts as values.  The new `dict` has 68 transcripts instead of 72.

In [13]:
import os
os.getcwd()

'c:\\Users\\adesi\\Documents\\GitHub\\kondrashov'

In [28]:
run kondrashov

#### Step 1

In [34]:
i = 0
vartranscripts = {}
for gene in loci:
    vt = get_transcripts_from_variants(gene, True)
    #i += len(vt)    
    print(gene, vt[0], len(vt), 'transcripts')
    vartranscripts.update({gene: vt})

ABCD1 NM_000033.4 1 transcripts
ALPL NM_000478.6 1 transcripts
AR NM_000044.6 1 transcripts
ATP7B NM_000053.4 1 transcripts
BTK NM_000061.3 1 transcripts
CASR NM_000388.4 1 transcripts
CBS NM_000071.3 1 transcripts
CFTR NM_000492.3 3 transcripts
CYBB NM_000397.4 1 transcripts
F7 NM_019616.4 5 transcripts
F8 NC_000023.11:g.155022771G>A 2 transcripts
F9 NC_000023.11:g.139530710G>C 4 transcripts
G6PD NM_001360016.2 3 transcripts
GALT NM_000155.4 1 transcripts
GBA1 NM_000157.4 1 transcripts
GJB1 NM_000166.6 2 transcripts
HBB NC_000011.10:g.5227147G>T 10 transcripts
HPRT1 NM_000194.3 3 transcripts
IL2RG NM_000206.3 1 transcripts
KCNH2 NM_000238.4 1 transcripts
KCNQ1 NM_000218.3 1 transcripts
L1CAM NM_001278116.2 1 transcripts
LDLR NM_000527.4 2 transcripts
MPZ NM_000530.8 1 transcripts
MYH7 NM_000257.4 1 transcripts
TYR NM_000372.5 1 transcripts
PAH NM_000277.3 1 transcripts
PMM2 NM_000303.3 2 transcripts
RHO NM_000539.3 1 transcripts
TP53 NM_000546.6 1 transcripts
TTR NM_000371.4 1 transcr

### Step 2
##### Extract human protein sequence records from the fasta files

Take the fasta files containing ortholog sequences, search for human sequences and extract the corresponding record.

**Output:** `protrecs` dictionary with genes as keys and lists of protein records as values. Unchanged.

In [35]:
i = 1
protrecs = {}
for gene in loci:
    protrecs.update({gene: []})
    for record in SeqIO.parse('fasta/{0}.fasta'.format(gene), 'fasta'):
        sp = get_species(record)
        if sp=='Homo sapiens':
            protrecs[gene].append(record)
            print(i, gene, record.id, len(record))
            i += 1

1 ABCD1 NP_000024.2 745
2 ABCD1 NP_001427676.1 845
3 ABCD1 XP_047297873.1 575
4 ABCD1 NP_002981.2 93
5 ABCD1 XP_047290405.1 106
6 ABCD1 XP_054169574.1 106
7 ABCD1 XP_054169575.1 93
8 ALPL NP_000469.3 524
9 ALPL NP_001120973.2 469
10 ALPL NP_001170991.1 447
11 ALPL XP_016856392.1 472
12 ALPL XP_054191723.1 472
13 AR NP_000035.2 920
14 AR NP_001011645.1 388
15 AR NP_001334990.1 644
16 AR NP_001334992.1 648
17 AR NP_001334993.1 572
18 AR NP_001648.1 252
19 AR NP_001333071.1 172
20 AR NP_001619.1 316
21 ATP7B NP_000044.2 1465
22 ATP7B NP_001005918.1 1258
23 ATP7B NP_001230111.1 1354
24 ATP7B NP_001317507.1 1387
25 ATP7B NP_001317508.1 1381
26 ATP7B NP_001393442.1 1463
27 ATP7B NP_001393443.1 1454
28 ATP7B NP_001393444.1 1447
29 ATP7B NP_001393446.1 1433
30 ATP7B NP_001393448.1 1420
31 ATP7B NP_001393449.1 1417
32 ATP7B NP_001393452.1 1404
33 ATP7B NP_001393453.1 1406
34 ATP7B NP_001393454.1 1400
35 ATP7B NP_001393455.1 1397
36 ATP7B NP_001393459.1 1385
37 ATP7B NP_001393463.1 1369
38 ATP7B

### STEP 3 
##### Match variant transcript IDs with ortholog proteins

Take variant transcript ID, query the NCBI database, extract the corresponding protein, and look for it in the ortholog set.  Flag problems.

**Output:** `translation` dictionary with transcript IDs as keys and lists of protein IDs as values.

In [36]:
#%%time
translation = {}
i = 1
for gene in loci:
    for transcript in vartranscripts[gene]:
        if transcript[:3] == 'NM_':
            rec = get_transcript(transcript)
            seq = get_CDS(rec)
            match = False
            for prot in protrecs[gene]: 
                if seq[4] == prot.seq:
                    print(i, gene, transcript, prot.id)
                    match = True
                    translation.update({transcript: prot.id})
            if not match:
                print(i, gene, transcript, 'no matching protein         <<<<<')
        else:
            print(i, gene, transcript, 'invalid transcript name         *****')
        i += 1

1 ABCD1 NM_000033.4 NP_000024.2
2 ALPL NM_000478.6 NP_000469.3
3 AR NM_000044.6 NP_000035.2
4 ATP7B NM_000053.4 NP_000044.2
5 BTK NM_000061.3 NP_000052.1
6 CASR NM_000388.4 NP_000379.3
7 CBS NM_000071.3 NP_000062.1
8 CFTR NM_000492.3 NP_000483.3
9 CFTR NC_000007.14:g.117675861C>T invalid transcript name         *****
10 CFTR NM_000492.4 NP_000483.3
11 CYBB NM_000397.4 NP_000388.2
12 F7 NM_019616.4 NP_062562.1
13 F7 NM_000131.4 NP_000122.1
14 F7 NM_000131.5 NP_000122.1
15 F7 NC_000013.11:g.113105748C>G invalid transcript name         *****
16 F7 NC_000013.11:g.113105784G>A invalid transcript name         *****
17 F8 NC_000023.11:g.155022771G>A invalid transcript name         *****
18 F8 NM_000132.4 NP_000123.1
19 F9 NC_000023.11:g.139530710G>C invalid transcript name         *****
20 F9 NM_000133.4 NP_000124.1
21 F9 NC_000023.11:g.139530731A>T invalid transcript name         *****
22 F9 NC_000023.11:g.139530716T>A invalid transcript name         *****
23 G6PD NM_001360016.2 NP_001035810

## clean up variants not matching any protein
* HBB 
* G6PD

In [ ]:
vardata = pd.read_csv('pathogenic/HBB.csv')
i = vardata['Name'].tolist().index('Hb Little Rock')
vardata[(vardata.index<i+3) & (vardata.index>i-3)]

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
14,30278,single nucleotide variant,NM_000518.5(HBB):c.82G>T (p.Ala28Ser),3043,HBB,HGNC:4827,Pathogenic,1,"Apr 07, 2026",35424040,...,A,-,-,-,-,-,-,SCV007592675,-,-
15,30280,single nucleotide variant,NM_000518.5(HBB):c.295G>A (p.Val99Met),3043,HBB,HGNC:4827,Pathogenic,1,"Aug 07, 2025",33933298,...,T,-,-,-,-,-,-,SCV000603896|SCV000919454|SCV002089199|SCV0040...,-,-
16,30289,single nucleotide variant,Hb Little Rock,3043,HBB,HGNC:4827,Pathogenic,1,"Jan 01, 1987",36020563,...,Y,-,-,-,-,-,-,SCV000763147,-,-
17,30290,single nucleotide variant,NM_000518.5(HBB):c.127T>C (p.Phe43Leu),3043,HBB,HGNC:4827,Likely pathogenic,1,"May 30, 2025",33924146,...,G,-,-,-,-,-,-,SCV007535976,-,-
18,30291,single nucleotide variant,NM_000518.5(HBB):c.89G>A (p.Gly30Asp),3043,HBB,HGNC:4827,Likely pathogenic,1,"Aug 07, 2024",35685286,...,T,-,-,-,-,-,-,SCV005625782,-,-


In [ ]:
vardata.loc[i, 'Name'] = 'NM_000518.5(HBB):c.432C>R (p.His144Gln)'
vardata.to_csv('pathogenic/HBB.csv', index=False)
vardata[(vardata.index<i+3) & (vardata.index>i-3)]

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
14,30278,single nucleotide variant,NM_000518.5(HBB):c.82G>T (p.Ala28Ser),3043,HBB,HGNC:4827,Pathogenic,1,"Apr 07, 2026",35424040,...,A,-,-,-,-,-,-,SCV007592675,-,-
15,30280,single nucleotide variant,NM_000518.5(HBB):c.295G>A (p.Val99Met),3043,HBB,HGNC:4827,Pathogenic,1,"Aug 07, 2025",33933298,...,T,-,-,-,-,-,-,SCV000603896|SCV000919454|SCV002089199|SCV0040...,-,-
16,30289,single nucleotide variant,NM_000518.5(HBB):c.432C>R (p.His144Gln),3043,HBB,HGNC:4827,Pathogenic,1,"Jan 01, 1987",36020563,...,Y,-,-,-,-,-,-,SCV000763147,-,-
17,30290,single nucleotide variant,NM_000518.5(HBB):c.127T>C (p.Phe43Leu),3043,HBB,HGNC:4827,Likely pathogenic,1,"May 30, 2025",33924146,...,G,-,-,-,-,-,-,SCV007535976,-,-
18,30291,single nucleotide variant,NM_000518.5(HBB):c.89G>A (p.Gly30Asp),3043,HBB,HGNC:4827,Likely pathogenic,1,"Aug 07, 2024",35685286,...,T,-,-,-,-,-,-,SCV005625782,-,-


### G6PD

In [ ]:
vardata = pd.read_csv('pathogenic/G6PD.csv')
i = vardata['Name'].tolist().index('G6PD ZURICH') 
vardata[(vardata.index<i+3) & (vardata.index>i-3)]

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
40,25455,single nucleotide variant,NM_000402.4(G6PD):c.298T>C (p.Tyr100His),2539,G6PD,HGNC:4057,Pathogenic/Likely pathogenic,1,"Nov 25, 2024",137852349,...,G,-,-,-,-,-,-,SCV001443096|SCV002599323|SCV003445326|SCV0038...,-,-
41,25456,single nucleotide variant,NM_000402.4(G6PD):c.683G>A (p.Arg228His),2539,G6PD,HGNC:4057,Pathogenic/Likely pathogenic,1,"Jan 02, 2026",137852332,...,T,-,-,-,-,-,-,SCV001142107|SCV002599340|SCV004299714|SCV0050...,-,-
42,25459,single nucleotide variant,G6PD ZURICH,2539,G6PD,HGNC:4057,Pathogenic,1,"Aug 12, 2022",2070350038,...,C,-,-,-,-,-,-,SCV002599384,-,-
43,25461,single nucleotide variant,NM_000402.4(G6PD):c.1466G>C (p.Arg489Pro),2539,G6PD,HGNC:4057,Pathogenic,1,"Feb 05, 2024",72554665,...,G,-,-,-,-,-,-,SCV000330997|SCV000914284|SCV001367223|SCV0013...,-,-
44,99398,single nucleotide variant,NM_001360016.2(G6PD):c.1360C>T (p.Arg454Cys),2539,G6PD,HGNC:4057,Pathogenic,1,"Apr 29, 2026",398123546,...,A,-,-,-,-,-,-,SCV000225263|SCV000281545|SCV000603769|SCV0006...,-,-


In [ ]:
vardata.loc[i, 'Name'] = 'NM_001360016.2(G6PD):c.1288-2A>G)'
vardata.to_csv('pathogenic/G6PD.csv', index=False)
vardata[(vardata.index<i+3) & (vardata.index>i-3)]

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
40,25455,single nucleotide variant,NM_000402.4(G6PD):c.298T>C (p.Tyr100His),2539,G6PD,HGNC:4057,Pathogenic/Likely pathogenic,1,"Nov 25, 2024",137852349,...,G,-,-,-,-,-,-,SCV001443096|SCV002599323|SCV003445326|SCV0038...,-,-
41,25456,single nucleotide variant,NM_000402.4(G6PD):c.683G>A (p.Arg228His),2539,G6PD,HGNC:4057,Pathogenic/Likely pathogenic,1,"Jan 02, 2026",137852332,...,T,-,-,-,-,-,-,SCV001142107|SCV002599340|SCV004299714|SCV0050...,-,-
42,25459,single nucleotide variant,NM_001360016.2(G6PD):c.1288-2A>G),2539,G6PD,HGNC:4057,Pathogenic,1,"Aug 12, 2022",2070350038,...,C,-,-,-,-,-,-,SCV002599384,-,-
43,25461,single nucleotide variant,NM_000402.4(G6PD):c.1466G>C (p.Arg489Pro),2539,G6PD,HGNC:4057,Pathogenic,1,"Feb 05, 2024",72554665,...,G,-,-,-,-,-,-,SCV000330997|SCV000914284|SCV001367223|SCV0013...,-,-
44,99398,single nucleotide variant,NM_001360016.2(G6PD):c.1360C>T (p.Arg454Cys),2539,G6PD,HGNC:4057,Pathogenic,1,"Apr 29, 2026",398123546,...,A,-,-,-,-,-,-,SCV000225263|SCV000281545|SCV000603769|SCV0006...,-,-


### STEP 4  
##### Extract individual variants

Process each ClinVar dataset and extract variants for each transcript.

**Output:** `variants` dictionary with transcript IDs as keys and lists of changes as values.

In [37]:
print('gene transcript protein changes errors')
for gene in loci:
    transcripts = vartranscripts[gene]
    for transcript in transcripts:
        sites, wilds, muts, errs = get_all_changes(gene, transcript, True)
        print(gene, transcript, translation[transcript], len(sites), len(errs))

gene transcript protein changes errors


KeyError: 'Accession'

Let's look more closely at the variants where there are errors.  **They appear to be variants where the name does not specify an amino acid change.**  We can ignore them.

In [7]:
print('gene transcript protein changes errors')
for gene in loci:
    transcripts = vartranscripts[gene]
    for transcript in transcripts:
        sites, wilds, muts, errs = get_all_changes(gene, transcript, True)
        if len(errs)>0:
            print(gene, transcript, translation[transcript], len(sites), len(errs))
            print('   ', errs)

gene transcript protein changes errors
ATP7B NM_000053.3 XP_005266487.1 0 2
    [('NM_000053.3(ATP7B):c.3809A>G', 'VCV000003859'), ('NM_000053.3(ATP7B):c.[3443T>C;3526G>A]', 'VCV000003863')]
BTK NM_001287345.1 NP_001274274.1 0 8
    [('NM_001287345.1(BTK):c.1039-1398T>C', 'VCV000571649'), ('NM_001287345.1(BTK):c.1039-1411G>T', 'VCV000656776'), ('NM_001287345.1(BTK):c.1039-1418C>A', 'VCV000011375'), ('NM_001287345.1(BTK):c.1038+1529A>G', 'VCV000011343'), ('NM_001287345.1(BTK):c.1038+1516C>A', 'VCV000011374'), ('NM_001287345.1(BTK):c.1038+1464T>C', 'VCV000011373'), ('NM_001287345.1(BTK):c.1038+813T>G', 'VCV000011371'), ('NM_001287345.1(BTK):c.1038+44A>G', 'VCV000011346')]
BTK NM_000061.3 NP_000052.1 25 1
    [('NM_000061.3(BTK):c.763C>T', 'VCV000011363')]
CASR NM_000388.4 NP_000379.3 108 1
    [('NM_000388.4(CASR):c.1609-2A>G', 'VCV000854168')]
CFTR NM_000492.3 NP_000483.3 225 7
    [('NM_000492.3(CFTR):c.[350G>A;1210-12[5]]', 'VCV000209047'), ('NM_000492.3(CFTR):c.[1075C>A;1079C>A]', 'V

We collect all valid variants in a `variants` dictionary.

In [8]:
variants = {}
for gene in loci:
    print(gene, end=' ... ')
    transcripts = vartranscripts[gene]
    for transcript in transcripts:
        sites, wilds, muts, errs = get_all_changes(gene, transcript, True)
        if len(sites)>0:
            variants.update({transcript: (sites, wilds, muts, errs)})

ABCD1 ... ALPL ... AR ... ATP7B ... BTK ... CASR ... CBS ... CFTR ... CYBB ... F7 ... F8 ... F9 ... G6PD ... GALT ... GBA ... GJB1 ... HBB ... HPRT1 ... IL2RG ... KCNH2 ... KCNQ1 ... L1CAM ... LDLR ... MPZ ... MYH7 ... TYR ... PAH ... PMM2 ... RHO ... TP53 ... TTR ... VWF ... 

In [9]:
sites, wilds, muts, errs = variants['NM_000518.4']
print(sites)
print(wilds)
print(muts)

[147, 147, 147, 143, 128, 125, 122, 111, 107, 107, 102, 101, 100, 100, 100, 100, 100, 100, 98, 95, 93, 93, 92, 90, 89, 83, 83, 83, 69, 68, 61, 46, 43, 37, 33, 31, 29, 27, 24, 9, 7]
['H', 'H', 'H', 'A', 'Q', 'P', 'E', 'L', 'L', 'L', 'E', 'P', 'D', 'D', 'D', 'D', 'D', 'D', 'H', 'D', 'H', 'H', 'L', 'S', 'L', 'K', 'K', 'K', 'L', 'V', 'V', 'F', 'F', 'P', 'L', 'R', 'L', 'E', 'V', 'K', 'E']
['P', 'L', 'D', 'D', 'R', 'R', 'K', 'P', 'R', 'Q', 'K', 'L', 'A', 'G', 'V', 'Y', 'H', 'N', 'L', 'H', 'N', 'Y', 'P', 'N', 'P', 'T', 'M', 'E', 'H', 'E', 'E', 'S', 'V', 'H', 'Q', 'T', 'Q', 'G', 'F', 'E', 'K']


## Extract variable sites

Process each alignment and extract variable sites for each protein.

**Output:** `variable` dictionary with transcript IDs as keys and dictionaries of variable sites as values.

In [10]:
variable = {}
for gene in loci:
    print(gene, end=' ... ')
    transcripts = vartranscripts[gene]
    for transcript in transcripts:
        protein = translation[transcript]
        variable.update({transcript: get_variable_sites(gene, protein, False)})

ABCD1 ... ALPL ... AR ... ATP7B ... BTK ... CASR ... CBS ... CFTR ... CYBB ... F7 ... F8 ... F9 ... G6PD ... GALT ... GBA ... GJB1 ... HBB ... HPRT1 ... IL2RG ... KCNH2 ... KCNQ1 ... L1CAM ... LDLR ... MPZ ... MYH7 ... TYR ... PAH ... PMM2 ... RHO ... TP53 ... TTR ... VWF ... 

In [11]:
!open .

## Identify putative CPDs

In [12]:
print('gene\ttranscript\tsite\twild\tmut\tvar')
final_transcripts = variants.keys()
for gene in loci:
    transcripts = vartranscripts[gene]
    for transcript in transcripts:
        if transcript in final_transcripts:
            sites, wilds, muts, errs = variants[transcript]
            tmp_variable = variable[transcript]
            variable_sites = tmp_variable.keys()
            n = len(sites)
            for i in range(n):
                site = sites[i]
                if (site in variable_sites) and (muts[i] in tmp_variable[site]):
                    print(gene, transcript, site, wilds[i], muts[i], tmp_variable[site], sep='\t')

gene	transcript	site	wild	mut	var
ALPL	NM_000478.6	491	G	R	['R']
AR	NM_000044.6	611	N	K	['E', 'K']
AR	NM_000044.6	788	M	V	['V', 'C', 'Y', 'I']
ATP7B	NM_000053.4	1178	T	A	['A']
ATP7B	NM_000053.4	969	R	Q	['Q']
CFTR	NM_000492.4	1	M	R	['L', 'P', 'Q', 'R']
CFTR	NM_000492.4	13	S	F	['G', 'F']
F8	NM_000132.3	2319	P	L	['L']
F8	NM_000132.3	2185	L	S	['S']
F8	NM_000132.3	2183	M	V	['G', 'V']
F8	NM_000132.3	2038	N	S	['H', 'K', 'S']
F8	NM_000132.3	1979	G	V	['V']
F8	NM_000132.3	585	I	T	['M', 'T']
F8	NM_000132.3	584	Q	K	['K', 'S', 'P']
F8	NM_000132.3	494	I	T	['T']
F9	NM_000133.3	75	R	Q	['Q']
GALT	NM_000155.4	23	T	A	['A', 'W']
GALT	NM_000155.4	97	N	D	['Q', 'D', 'E']
GALT	NM_000155.4	114	H	L	['L']
GALT	NM_000155.4	129	M	L	['L']
GALT	NM_000155.4	186	H	Y	['Y']
GALT	NM_000155.4	198	I	T	['D', 'T', 'V']
GALT	NM_000155.4	204	R	P	['P']
GALT	NM_000155.4	212	Q	P	['P']
GALT	NM_000155.4	226	L	P	['P']
GALT	NM_000155.4	319	H	Q	['Q']
GALT	NM_000155.4	329	S	P	['P']
GALT	NM_000155.4	333	R	L	['L', 'S']
GALT	NM_000155.4	3

## Look at particular sites

The following command allows you to double check particular sites.  It now also outputs which site each species has.

In [13]:
get_alleles('ATP7B', translation['NM_000053.4'], 969, True)

     Gene: ATP7B
    Human: 969 R
Alignment: 1102 R
R    105
-      6
Q      1
dtype: int64

R XP_012668202.1 Otolemur garnettii
R XP_023363838.1 Otolemur garnettii
R XP_023363839.1 Otolemur garnettii
R XP_012638114.1 Microcebus murinus
R XP_012638117.1 Microcebus murinus
R XP_020145970.1 Microcebus murinus
R XP_012638108.1 Microcebus murinus
R XP_012504533.1 Propithecus coquereli
R XP_012504538.1 Propithecus coquereli
R XP_012504534.1 Propithecus coquereli
R XP_012504529.1 Propithecus coquereli
R XP_008070488.1 Carlito syrichta
R XP_008070485.1 Carlito syrichta
R XP_008070483.1 Carlito syrichta
- XP_008070484.1 Carlito syrichta
R XP_008070486.1 Carlito syrichta
R XP_021561720.1 Carlito syrichta
R XP_008070482.1 Carlito syrichta
R XP_011920046.1 Cercocebus atys
R XP_016780802.1 Pan troglodytes
R XP_005266488.1 Homo sapiens
R XP_016876116.1 Homo sapiens
R NP_001005918.1 Homo sapiens
R NP_001230111.1 Homo sapiens
R NP_001317507.1 Homo sapiens
R NP_001317508.1 Homo sapiens
R XP_005266487.

(True,
 969,
 'R',
 R    105
 -      6
 Q      1
 dtype: int64)

In [14]:
get_alleles('F8', translation['NM_000132.3'], 2038, True)

     Gene: F8
    Human: 2038 N
Alignment: 2964 N
N    46
-    14
H     4
S     3
K     3
dtype: int64

N XP_021524500.1 Aotus nancymaae
K XP_003802782.1 Otolemur garnettii
K XP_012517042.1 Propithecus coquereli
K XP_020140823.1 Microcebus murinus
N XP_017376032.2 Cebus imitator
N XP_032121193.1 Sapajus apella
N XP_032121192.1 Sapajus apella
N XP_032121191.1 Sapajus apella
N XP_032121190.1 Sapajus apella
N XP_003943974.1 Saimiri boliviensis boliviensis
S XP_035144731.1 Callithrix jacchus
S XP_035144732.1 Callithrix jacchus
S XP_035144729.1 Callithrix jacchus
- XP_008960582.2 Pan paniscus
- XP_030861623.1 Gorilla gorilla gorilla
- XP_030662513.1 Nomascus leucogenys
N XP_032612179.1 Hylobates moloch
N XP_003279379.1 Nomascus leucogenys
- NP_063916.1 Homo sapiens
N NP_000123.1 Homo sapiens
N XP_034806192.1 Pan paniscus
N XP_003810005.2 Pan paniscus
N XP_034806193.1 Pan paniscus
N XP_016800115.1 Pan troglodytes
N XP_009438145.1 Pan troglodytes
N XP_016800113.1 Pan troglodytes
N XP_01680011

(True,
 2038,
 'N',
 N    46
 -    14
 H     4
 S     3
 K     3
 dtype: int64)

In [18]:
translation['NM_000157.4']

'NP_001005742.1'

In [15]:
get_alleles('TTR', translation['NM_000371.3'], 142, True)

     Gene: TTR
    Human: 142 V
Alignment: 144 V
V    26
-     2
L     1
I     1
dtype: int64

L XP_011788672.1 Colobus angolensis palliatus
- XP_010333769.1 Saimiri boliviensis boliviensis
V XP_010333768.1 Saimiri boliviensis boliviensis
V XP_032149005.1 Sapajus apella
V XP_012302669.1 Aotus nancymaae
V NP_001254679.1 Callithrix jacchus
V XP_003784812.1 Otolemur garnettii
V XP_012638014.1 Microcebus murinus
V XP_012512049.1 Propithecus coquereli
V NP_001127064.1 Pongo abelii
V NP_000362.1 Homo sapiens
V XP_004059337.1 Gorilla gorilla gorilla
V XP_032008476.1 Hylobates moloch
V XP_003830303.2 Pan paniscus
V NP_001009137.1 Pan troglodytes
V XP_011930274.1 Cercocebus atys
I XP_007972558.2 Chlorocebus sabaeus
V XP_011829762.1 Mandrillus leucophaeus
V XP_025220794.1 Theropithecus gelada
V XP_025220793.1 Theropithecus gelada
V XP_023063511.2 Piliocolobus tephrosceles
V XP_017713537.1 Rhinopithecus bieti
V XP_010379114.1 Rhinopithecus roxellana
V XP_011788674.1 Colobus angolensis palliatus
V

(True,
 142,
 'V',
 V    26
 -     2
 L     1
 I     1
 dtype: int64)

In [16]:
get_alleles('GBA', translation['NM_000157.4'], 535, True)

     Gene: GBA
    Human: 535 R
Alignment: 547 R
R    36
L     3
H     1
-     1
dtype: int64

- XP_037855735.1 Chlorocebus sabaeus
R XP_012514200.1 Propithecus coquereli
R XP_012623306.1 Microcebus murinus
R XP_012623305.1 Microcebus murinus
R XP_023372167.1 Otolemur garnettii
R XP_012664043.1 Otolemur garnettii
R XP_008048281.1 Carlito syrichta
R XP_021564251.1 Carlito syrichta
R XP_008048283.1 Carlito syrichta
R XP_011932225.1 Cercocebus atys
R XP_021528768.1 Aotus nancymaae
R XP_039315825.1 Saimiri boliviensis boliviensis
R XP_010346697.1 Saimiri boliviensis boliviensis
R XP_032129098.1 Sapajus apella
R XP_017368970.1 Cebus imitator
H XP_032010219.1 Hylobates moloch
R XP_030866469.1 Gorilla gorilla gorilla
R XP_030866465.1 Gorilla gorilla gorilla
R NP_001165282.1 Homo sapiens
R NP_001165283.1 Homo sapiens
R NP_001005742.1 Homo sapiens
R XP_030680202.1 Nomascus leucogenys
R XP_024098242.1 Pongo abelii
R NP_001127488.1 Pongo abelii
L XP_005541629.1 Macaca fascicularis
L XP_005541627.

(True,
 535,
 'R',
 R    36
 L     3
 H     1
 -     1
 dtype: int64)

In [17]:
get_alleles('GALT', translation['NM_000155.4'], 23, True)

     Gene: GALT
    Human: 23 T
Alignment: 43 T
T    37
-    24
W     1
A     1
dtype: int64

T XP_035106695.1 Callithrix jacchus
T XP_008068994.1 Carlito syrichta
- XP_034823261.1 Pan paniscus
T XP_034823259.1 Pan paniscus
T XP_034823258.1 Pan paniscus
T XP_034823256.1 Pan paniscus
- XP_023059055.1 Piliocolobus tephrosceles
- XP_037854236.1 Chlorocebus sabaeus
- XP_033094123.1 Trachypithecus francoisi
- XP_017747549.1 Rhinopithecus bieti
- XP_028690929.1 Macaca mulatta
- XP_011805906.1 Colobus angolensis palliatus
- NP_001245261.1 Homo sapiens
- XP_034823265.1 Pan paniscus
T XP_011924614.1 Cercocebus atys
W XP_011924616.1 Cercocebus atys
T XP_011924615.1 Cercocebus atys
- XP_011924617.1 Cercocebus atys
T XP_021783112.2 Papio anubis
T XP_025214557.1 Theropithecus gelada
T XP_021783108.2 Papio anubis
- XP_021783109.2 Papio anubis
- XP_021783111.1 Papio anubis
T XP_009186860.2 Papio anubis
T XP_033094120.1 Trachypithecus francoisi
- XP_033094122.1 Trachypithecus francoisi
- XP_023059054.

(True,
 23,
 'T',
 T    37
 -    24
 W     1
 A     1
 dtype: int64)

In [18]:
get_alleles('F9', translation['NM_000133.3'], 75, True)

     Gene: F9
    Human: 75 R
Alignment: 75 R
R    39
Q     1
dtype: int64

R XP_002763378.1 Callithrix jacchus
R XP_012325320.1 Aotus nancymaae
R XP_017355402.1 Cebus imitator
R XP_032124214.1 Sapajus apella
R XP_003940777.1 Saimiri boliviensis boliviensis
R XP_011930388.1 Cercocebus atys
R XP_037844800.1 Chlorocebus sabaeus
R XP_007991031.1 Chlorocebus sabaeus
R NP_001103153.1 Macaca mulatta
R XP_028697500.1 Macaca mulatta
R XP_011739181.1 Macaca nemestrina
R XP_028697499.1 Macaca mulatta
R XP_025228623.1 Theropithecus gelada
R XP_025228622.1 Theropithecus gelada
R XP_017809908.1 Papio anubis
R XP_003918402.1 Papio anubis
R XP_023058339.1 Piliocolobus tephrosceles
R XP_023058338.1 Piliocolobus tephrosceles
R XP_033057437.1 Trachypithecus francoisi
R XP_010387043.1 Rhinopithecus roxellana
R XP_011816431.1 Colobus angolensis palliatus
R XP_011816430.1 Colobus angolensis palliatus
R XP_009233598.2 Pongo abelii
R XP_002832230.2 Pongo abelii
R XP_032612351.1 Hylobates moloch
R XP_03261235

(True,
 75,
 'R',
 R    39
 Q     1
 dtype: int64)

In [14]:
translation['NM_000527.4']

'NP_000518.1'

In [13]:
get_alleles('LDLR', translation['NM_000527.4'], 215, True)

     Gene: LDLR
    Human: 215 R
Alignment: 351 R
R    35
-     9
H     9
dtype: int64

R XP_020139390.1 Microcebus murinus
R XP_012496802.1 Propithecus coquereli
H XP_021573646.1 Carlito syrichta
H XP_023373820.1 Otolemur garnettii
H XP_012665677.1 Otolemur garnettii
H XP_021532454.1 Aotus nancymaae
H XP_035141328.1 Callithrix jacchus
H XP_002761791.1 Callithrix jacchus
H XP_039321728.1 Saimiri boliviensis boliviensis
H XP_037583448.1 Cebus imitator
H XP_032107631.1 Sapajus apella
R XP_011810343.1 Colobus angolensis palliatus
R XP_032025606.1 Hylobates moloch
R XP_030676686.1 Nomascus leucogenys
R XP_024092494.1 Pongo abelii
R XP_024092493.1 Pongo abelii
- XP_024206936.1 Pan troglodytes
- NP_001182729.1 Homo sapiens
R NP_001182728.1 Homo sapiens
- NP_001182732.1 Homo sapiens
R XP_011526312.1 Homo sapiens
R NP_001182727.1 Homo sapiens
R NP_000518.1 Homo sapiens
R XP_018871452.1 Gorilla gorilla gorilla
R XP_018871450.1 Gorilla gorilla gorilla
- XP_034800282.1 Pan paniscus
R XP_034800281

(True,
 215,
 'R',
 R    35
 -     9
 H     9
 dtype: int64)

# Validate CPDs

Here we test whether a putative CPD is valid by investigating the vicinity of the sequence, 10 amino acids on either side of the site.

This function pulls out the species and protein IDs showing a particular amino acid at a site.

In [19]:
get_species_with_allele('GBA', translation['NM_000157.4'], 535, 'L')

[('Macaca fascicularis', 'XP_005541629.1'),
 ('Macaca fascicularis', 'XP_005541627.1'),
 ('Macaca fascicularis', 'XP_005541626.1')]

In [20]:
get_species_with_allele('GALT', translation['NM_000155.4'], 23, 'A')

[('Otolemur garnettii', 'XP_003800307.1')]

This function compares the human and nonhuman sequences in the -10 / +10 range.  The output at the bottom shows the numbers of:

* differences between the sequences
* gaps 
* sites out of range

For example, the GBA site at the end means that 9 sites are outside the range.  The GALT sequence shows some aminoacid differences. 

In [21]:
local_compare_to_human('GBA', translation['NM_000157.4'], 535, 'XP_005541629.1')

-10 G G
-9 Y Y
-8 S S
-7 I I
-6 H H
-5 T T
-4 Y Y
-3 L L
-2 W W
-1 R C
0 R L
1 Q Q
2 outside sequence
3 outside sequence
4 outside sequence
5 outside sequence
6 outside sequence
7 outside sequence
8 outside sequence
9 outside sequence
10 outside sequence


(1, 0, 9)

In [22]:
local_compare_to_human('GALT', translation['NM_000155.4'], 23, 'XP_003800307.1')

-10 Q Q
-9 A A
-8 S S
-7 E E
-6 A A
-5 D D
-4 A I
-3 A P
-2 A V
-1 A A
0 T A
1 F F
2 R Q
3 A A
4 N S
5 D D
6 H H
7 Q Q
8 H H
9 I I
10 R R


(5, 0, 0)

This function spits out a site by site comparison of the human and nonhuman sequences.  Only the differences are shown.

In [23]:
global_compare_to_human('GBA', translation['NM_000157.4'], 'XP_005541629.1')

5 S -
6 S -
7 P -
10 E K
16 L S
17 S G
24 G A
67 F L
68 D E
70 P L
94 M T
96 P T
101 H R
213 R H
344 V -
345 L -
346 T -
347 D -
348 P -
349 E -
350 A -
351 A -
352 K -
353 Y -
354 V -
355 H -
356 G -
357 I -
358 A -
359 V -
360 H -
361 W -
362 Y -
363 L -
364 D -
365 F -
366 L -
367 A -
368 P -
369 A -
370 K -
371 A -
372 T -
373 L -
374 G -
375 E -
376 T -
377 H -
378 R -
379 L -
380 F -
381 P -
382 N -
383 T -
384 M -
385 L -
386 F -
387 A -
388 S -
389 E -
390 A -
391 C -
392 V -
393 G -
394 S -
395 K -
396 F -
397 W -
398 E -
399 Q -
401 S -
402 V -
403 R -
404 L -
405 G -
406 S -
407 W -
408 D -
409 R -
410 G -
411 M -
412 Q -
413 Y -
414 S -
415 H -
416 S -
417 I -
418 I -
419 T -
452 I V
496 A T
507 A T
546 R C
547 R L


In [24]:
global_compare_to_human('GALT', translation['NM_000155.4'], 'XP_003800307.1')

23 R C
26 T A
28 P L
29 Q D
30 Q R
39 A I
40 A P
41 A V
43 T A
45 R Q
47 N S
106 I T
115 Q H
118 S A
153 K E
154 S A
177 V M
203 A V
216 A S
248 Q R
251 K Q
256 E V
268 L I
269 R K
289 T V
310 P S
360 E A
363 A D
366 N D
669 H C
671 G R
828 T A


In [25]:
get_species_with_allele('CFTR', translation['NM_000492.4'], 13, 'F')

[('Otolemur garnettii', 'XP_003789873.1')]

In [26]:
local_compare_to_human('CFTR', translation['NM_000492.4'], 13, 'XP_003789873.1')

-10 P P
-9 L L
-8 E K
-7 K K
-6 A L
-5 S N
-4 V L
-3 V L
-2 - -
-1 - -
0 S F
1 K F
2 L L
3 F Y
4 F F
5 S S
6 W W
7 T T
8 R R
9 P P
10 I I


(7, 0, 0)

In [28]:
translation['NM_000132.3']

'NP_000123.1'

In [29]:
get_species_with_allele('F8', 'NP_000123.1', 2183, 'V')

[('Hylobates moloch', 'XP_032612179.1')]

In [30]:
local_compare_to_human('F8', 'NP_000123.1', 2183, 'XP_003789873.1')

UnboundLocalError: local variable 'nonaln' referenced before assignment